# MyoMap AI — One-Run Train (Robust)
No manual path edits. Auto-finds code + competition data. Internet ON, GPU P100.
1. Add Inputs: `myomap-ai-rsnaknee-code` + `rsna-knee-abnormality-detection` competition
2. Run All → Done

In [ ]:
# Cell 1 — Auto-locate code (no hard path)
import pathlib, shutil, sys
from pathlib import Path
found = list(Path("/kaggle/input").rglob("train.py"))
print("found train.py:", found)
src_root = found[0].parents[1] if found else None
print("src_root:", src_root)
dst = Path("/kaggle/working/myomap-ai")
if dst.exists():
    shutil.rmtree(dst)
if src_root:
    shutil.copytree(src_root, dst)
    print("copied to", dst, "files:", len(list(dst.rglob("*.py"))))
else:
    print("ERROR: code dataset not found — Add Input -> myomap-ai-rsnaknee-code")
print("train.py exists:", (dst/"src/train.py").exists())


In [ ]:
# Cell 2 — Check competition data (auto-search)
from pathlib import Path
import os
print("All inputs:")
for p in Path("/kaggle/input").iterdir():
    print(p, "->", [x.name for x in p.iterdir()][:5] if p.is_dir() else "")
    for q in p.rglob("train.csv"):
        print("  found train.csv:", q)
csvs = list(Path("/kaggle/input").rglob("train.csv"))
print("train.csv count:", len(csvs))
if not csvs:
    print("ERROR: competition data missing — Add Input -> RSNA Knee Abnormality Detection")
else:
    import pandas as pd
    df = pd.read_csv(csvs[0])
    print(df.head())
    print(df.shape)
    print(df.columns.tolist()[:20])


In [ ]:
# Cell 3 — Install (Internet ON, skip failing builds)
!pip install -q timm==1.0.9 transformers==4.44.2 albumentations==1.4.0 pydicom==2.4.4 pylibjpeg==1.4.0 accelerate==0.33.0 opencv-python-headless==4.10.0.84 2>&1 | tail -n 5
print("deps done — ignore numpy>=2 warnings")


In [ ]:
# Cell 4 — Train (no manual PYTHONPATH, no sed)
!PYTHONPATH=/kaggle/working/myomap-ai/src:$PYTHONPATH python /kaggle/working/myomap-ai/src/train.py --config /kaggle/working/myomap-ai/configs/config.yaml --fold 0 2>&1 | tee /kaggle/working/train.log
print("train finished — check /kaggle/working/train.log")


In [ ]:
# Cell 5 — Submission (optional)
!PYTHONPATH=/kaggle/working/myomap-ai/src:$PYTHONPATH python /kaggle/working/myomap-ai/scripts/submission.py --ckpt /kaggle/working/myomap-ai/models/best_fold0.pth --config /kaggle/working/myomap-ai/configs/config.yaml --out /kaggle/working/submission.csv 2>&1 | tail -n 20
!head /kaggle/working/submission.csv 2>&1 | head
!wc -l /kaggle/working/submission.csv 2>&1 | head
